In [3]:
import pandas as pd
df = pd.read_csv('Cleaned_Dataset.csv')

recommendation_df= df.copy()

print('=' * 50)
print(recommendation_df.head())
print(recommendation_df.shape)
print(recommendation_df.columns.tolist())




recommendation_df['listed_in'] = recommendation_df['listed_in'].fillna("")
recommendation_df['type'] = recommendation_df['type'].fillna("")
recommendation_df['title'] = recommendation_df['title'].fillna("")

recommendation_df['content'] = (
    recommendation_df['listed_in'] + " " +
    recommendation_df['type']
)

recommendation_df['content'] = recommendation_df['content'].str.lower()

recommendation_df['content'] = (
    recommendation_df['content']
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(recommendation_df[['title', 'listed_in', 'type', 'content']].head())

   Unnamed: 0 show_id     type                             title  \
0           0      s1    Movie              Dick Johnson Is Dead   
1           1      s3  TV Show                         Ganglands   
2           2      s6  TV Show                     Midnight Mass   
3           3     s14    Movie  Confessions of an Invisible Girl   
4           4      s8    Movie                           Sankofa   

          director        country date_added  release_year rating  duration  \
0  Kirsten Johnson  United States  9/25/2021          2020  PG-13    90 min   
1  Julien Leclercq         France  9/24/2021          2021  TV-MA  1 Season   
2    Mike Flanagan  United States  9/24/2021          2021  TV-MA  1 Season   
3    Bruno Garotti         Brazil  9/22/2021          2021  TV-PG    91 min   
4     Haile Gerima  United States  9/24/2021          1993  TV-MA   125 min   

                                           listed_in  
0                                      Documentaries  
1  Cri

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(recommendation_df['content'])

print('TF-IDF Matrix Shape:')
print(tfidf_matrix.shape)

print('\nNumber of features:')
print(len(tfidf.get_feature_names_out()))

TF-IDF Matrix Shape:
(8790, 45)

Number of features:
45


In [18]:
def get_recommendations(title, similarity_matrix, df, top_n=5):
    
    # Find the row position of the selected title
    matches = df.index[df['title'].str.lower() == title.lower()].tolist()
    
    if not matches:
        print(f"Title '{title}' was not found in the dataset.")
        return None
    
    title_index = matches[0]

    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[title_index]))

    # Sort from highest to lowest similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected title itself
    similarity_scores = similarity_scores[1:top_n + 1]

    # Get recommended row positions
    recommended_indices = [i[0] for i in similarity_scores]

    # Select recommendations
    recommendations = df.iloc[recommended_indices].copy()

    # Add similarity scores
    recommendations['similarity_score'] = [
        score for _, score in similarity_scores
    ]

    return recommendations

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all Netflix titles
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix created successfully!")
print("Shape:", similarity_matrix.shape)

Similarity matrix created successfully!
Shape: (8790, 8790)


In [7]:
print(recommendation_df.columns.tolist())

['Unnamed: 0', 'show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'content']


In [19]:
recommendations = get_recommendations(
    "Ganglands",
    similarity_matrix,
    recommendation_df,
    top_n=5
)

recommendations

,Unnamed: 0,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,content,similarity_score
24,24,s12,TV Show,Bangkok Breaking,Kongkiat Komesiri,Not Given,9/23/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...","crime tv shows, international tv shows, tv act...",1.0
390,390,s3977,TV Show,The Eagle of El-Se'eed,Not Given,Pakistan,3/25/2019,2018,TV-14,1 Season,"Crime TV Shows, International TV Shows, TV Act...","crime tv shows, international tv shows, tv act...",1.0
486,486,s5195,TV Show,The Truth,Not Given,Pakistan,11/1/2017,2008,TV-14,1 Season,"Crime TV Shows, International TV Shows, TV Act...","crime tv shows, international tv shows, tv act...",1.0
576,576,s6742,TV Show,Fatal Destiny,Not Given,Pakistan,7/13/2018,2016,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...","crime tv shows, international tv shows, tv act...",1.0
1177,1177,s1224,TV Show,Dealer,"Ange Basterga, Nicolas Lopez",France,3/10/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...","crime tv shows, international tv shows, tv act...",1.0


In [20]:
def precision_at_k(title, recommendations, df, k=5):

    selected = df[df['title'].str.lower() == title.lower()]

    if selected.empty:
        print(f"Title '{title}' was not found")
        return None

    selected_categories = set(
        str(selected.iloc[0]['listed_in']).lower().split(',')
    )

    relevant = 0

    for _, row in recommendations.head(k).iterrows():
        recommended_categories = set(
            str(row['listed_in']).lower().split(',')
        )

        selected_clean = {x.strip() for x in selected_categories}
        recommended_clean = {x.strip() for x in recommended_categories}

        if selected_clean.intersection(recommended_clean):
            relevant +=1

    precision = relevant / k

    return precision

In [21]:
ganglands_recommendations = get_recommendations(
    "Ganglands",
    similarity_matrix,
    recommendation_df,
    top_n=5
)

precision = precision_at_k(
    "Ganglands",
    ganglands_recommendations,
    recommendation_df,
    k=5
)

print(f"Precision@5: {precision:.2%}")

Precision@5: 100.00%


In [22]:
selected_titles = [
    "Ganglands",
    "Midnight Mass",
    "Sankofa",
    "Dick Johnson Is Dead"
]

results = []

for title in selected_titles:

    recommendations = get_recommendations(
        title,
        similarity_matrix,
        recommendation_df,
        top_n=5
    )

    precision = precision_at_k(
        title,
        recommendations,
        recommendation_df,
        k=5
    )

    results.append({
        "Title": title,
        "Precision@5": precision
    })

evaluation_df = pd.DataFrame(results)

evaluation_df

,Title,Precision@5
0,Ganglands,1.0
1,Midnight Mass,1.0
2,Sankofa,1.0
3,Dick Johnson Is Dead,1.0
